In [1]:
import sys
from pathlib import Path

# Walk upward from the current working directory until we find the
# repository root. We define the repo root as the folder that contains
# both `src/` (our Python package) and `data/` (our datasets).
#
# This is necessary because Jupyter in VS Code / Codespaces often runs
# with cwd = /.../notebooks instead of the project root.

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").exists() and (p / "data").exists():
            return p
    raise RuntimeError(f"Could not find repo root above {start}")

# Determine the true repository root regardless of where the notebook
# kernel was launched from

REPO_ROOT = find_repo_root(Path.cwd())

# Add the repo root to Python's import search path so that
# `import src.io` works inside notebooks

sys.path.insert(0, str(REPO_ROOT))

# Build an absolute path to the raw data directory so we never rely on
# fragile relative paths like "data/raw"

RAW_DIR = REPO_ROOT / "data" / "raw"
RAW_DIR = RAW_DIR.resolve()

# Now that paths and imports are stable, we can safely import libraries
# and our project code

import pandas as pd
from src.io import ingest_raw_csvs

# Display for sanity checking
REPO_ROOT, RAW_DIR

(PosixPath('/workspaces/protected-bike-lanes-ridership'),
 PosixPath('/workspaces/protected-bike-lanes-ridership/data/raw'))

In [2]:
import os

# Show the actual current working directory of the notebook kernel.
# In VS Code / Codespaces this is usually /.../notebooks instead of the repo root.
print("cwd:", os.getcwd())

# Show the repository root we detected via the bootstrap logic.
# This should point to /workspaces/protected-bike-lanes-ridership
print("repo_root:", REPO_ROOT)

# Further sanity check that the src/ package is actually present at the repo root.
# If this is False, imports like `from src.io import ...` will fail.
print("src exists:", (REPO_ROOT / "src").exists())

# Further sanity check that the data/ directory is present.
# If this is False, RAW_DIR construction is broken.
print("data exists:", (REPO_ROOT / "data").exists())

# Show the first entry on Python's module search path.
# This should be the repo root, meaning Python can find `src/` as a package.
print("sys.path[0]:", sys.path[0])

cwd: /workspaces/protected-bike-lanes-ridership/notebooks
repo_root: /workspaces/protected-bike-lanes-ridership
src exists: True
data exists: True
sys.path[0]: /workspaces/protected-bike-lanes-ridership


In [3]:
# Run the raw CSV ingest pipeline against the raw data directory.
# This reads every CSV under data/raw/, applies any safe parsing logic,
# and combines them into a single pandas DataFrame.
result = ingest_raw_csvs(RAW_DIR)

# The unified dataframe of all counter data
df = result.df

# Basic sanity checks so we know we didn't silently fail
print("Rows:", len(df))                      # total number of records loaded
print("Columns:", len(df.columns))           # how wide the dataset is
print("Files read:", len(result.files_read)) # how many CSVs were successfully ingested
print("Files failed:", len(result.files_failed))  # how many CSVs could not be read

# Peek at the first few rows to visually confirm the structure
df.head()

/workspaces/protected-bike-lanes-ridership/src/io.py:44: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, encoding="utf-8")


Rows: 838202
Columns: 38
Files read: 10
Files failed: 0


,date,2nd_ave_cycletrack,bicyclists_northbound,bicyclists_southbound,scooterist_northbound,scooterist_southbound,source_file,39th_ave_ne_greenway_at_ne_62nd_st_total,north,south,...,pedestrian_west,pedestrian_east,bike_east,bike_west,scooter_east,scooter_west,nw_58th_st_greenway_st_22nd_ave_nw_total,east,west,spokane_st._bridge_total
0,2015 Jan 01 12:00:00 AM,3.0,0.0,3.0,NaN,NaN,/workspaces/protected-bike-lanes-ridership/dat...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015 Jan 01 01:00:00 AM,9.0,0.0,9.0,NaN,NaN,/workspaces/protected-bike-lanes-ridership/dat...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015 Jan 01 02:00:00 AM,2.0,0.0,2.0,NaN,NaN,/workspaces/protected-bike-lanes-ridership/dat...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015 Jan 01 03:00:00 AM,0.0,0.0,0.0,NaN,NaN,/workspaces/protected-bike-lanes-ridership/dat...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015 Jan 01 04:00:00 AM,0.0,0.0,0.0,NaN,NaN,/workspaces/protected-bike-lanes-ridership/dat...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# List all column names in sorted order so we can see the full schema
# and spot things like multiple date fields, direction columns, or
# inconsistently named variables across files.
pd.Series(df.columns).sort_values()

1                                    2nd_ave_cycletrack
7              39th_ave_ne_greenway_at_ne_62nd_st_total
13                           bgt_north_of_ne_70th_total
22                                 bicyclist_northbound
23                                 bicyclist_southbound
2                                 bicyclists_northbound
3                                 bicyclists_southbound
30                                            bike_east
16                                           bike_north
17                                           bike_south
31                                            bike_west
10       broadway_cycle_track_north_of_e_union_st_total
18              chief_sealth_trl_north_of_thistle_total
0                                                  date
35                                                 east
19       elliott_bay_trail_in_myrtle_edwards_park_total
26    fremont_bridge_sidewalks,_south_of_n_34th_st_c...
25    fremont_bridge_sidewalks,_south_of_n_34th_

In [5]:
# Identify any columns that look like date or time fields.
# Seattle data often includes multiple timestamp columns, so this
# helps us see our candidates before deciding which one to use
# for daily aggregation.
[c for c in df.columns if "date" in c or "time" in c]

['date']

In [6]:
# Compute the fraction of missing values in each column and show
# the 20 most incomplete fields. This tells us which columns are
# mostly empty metadata and which ones are reliable enough to use
# for analysis and modeli
df.isna().mean().sort_values(ascending=False).head(20)

scooter_east                                      0.984050
scooter_west                                      0.984050
scooterist_northbound                             0.984024
scooterist_southbound                             0.984024
chief_sealth_trl_north_of_thistle_total           0.965198
39th_ave_ne_greenway_at_ne_62nd_st_total          0.953877
south                                             0.953877
north                                             0.953877
sb                                                0.924185
nb                                                0.924185
broadway_cycle_track_north_of_e_union_st_total    0.924185
nw_58th_st_greenway_st_22nd_ave_nw_total          0.912268
ped_north                                         0.910820
ped_south                                         0.910820
pedestrian_west                                   0.904382
pedestrian_east                                   0.904382
bike_west                                         0.9026

In [7]:
# Confirm that the key numeric fields we care about (north/south/east/west/nb/sb) are present
# and have reasonable summary statistics (e.g. no negative counts, no huge outliers)
df[["north","south","east","west","nb","sb"]].describe(include="all")

,north,south,east,west,nb,sb
count,38660.000000,38660.000000,180544.000000,180544.000000,63548.000000,63548.000000
mean,4.105251,4.547181,10.455379,9.835824,5.601026,6.314550
std,6.600789,6.840137,20.163672,19.955437,6.602593,6.716115
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000
50%,2.000000,2.000000,3.000000,3.000000,4.000000,4.000000
75%,5.000000,6.000000,12.000000,10.000000,8.000000,9.000000
max,60.000000,60.000000,940.000000,367.000000,99.000000,132.000000


In [8]:
# Check the fraction of missing values in the key numeric fields. If these are mostly empty, we may need to reconsider our analysis plan or do some imputation.
df[["north","south","east","west","nb","sb"]].isna().mean().sort_values()

west     0.784606
east     0.784606
sb       0.924185
nb       0.924185
north    0.953877
south    0.953877
dtype: float64

In [9]:
# Define column groups by bike/ped/scooter counters to see if the above summary stats include one or more of these groups
bike_cols = [
    "bike_north", "bike_south", "bike_east", "bike_west",
    "bicyclist_northbound", "bicyclist_southbound",
    "bicyclists_northbound", "bicyclists_southbound",
    "north", "south", "east", "west", "nb", "sb",
]

ped_cols = [
    "ped_north", "ped_south",
    "pedestrian_east", "pedestrian_west",
    "pedestrian_northbound", "pedestrian_southbound",
]

scooter_cols = [
    "scooter_east", "scooter_west",
    "scooterist_northbound", "scooterist_southbound",
]

# keep only columns that actually exist (defensive)
bike_cols = [c for c in bike_cols if c in df.columns]
ped_cols = [c for c in ped_cols if c in df.columns]
scooter_cols = [c for c in scooter_cols if c in df.columns]

bike_present = df[bike_cols].notna().any(axis=1)
bike_rows = df.loc[bike_present].copy()

len(df), len(bike_rows)

(838202, 678774)

In [10]:
# Check for overlap between bike rows and ped/scooter fields.
ped_present_on_bike_rows = bike_rows[ped_cols].notna().any(axis=1) if ped_cols else pd.Series(False, index=bike_rows.index)
scooter_present_on_bike_rows = bike_rows[scooter_cols].notna().any(axis=1) if scooter_cols else pd.Series(False, index=bike_rows.index)

print("Bike rows:", len(bike_rows))
print("Bike rows w/ any ped fields present:", int(ped_present_on_bike_rows.sum()))
print("Bike rows w/ any scooter fields present:", int(scooter_present_on_bike_rows.sum()))
print("Bike rows w/ ped OR scooter present:", int((ped_present_on_bike_rows | scooter_present_on_bike_rows).sum()))


Bike rows: 678774
Bike rows w/ any ped fields present: 249341
Bike rows w/ any scooter fields present: 26760
Bike rows w/ ped OR scooter present: 262732


In [11]:
# Identify which sensors are mixing modes
mixed = bike_rows.loc[ped_present_on_bike_rows | scooter_present_on_bike_rows, ["source_file", "date"] + bike_cols + ped_cols + scooter_cols]

# Count mixed rows per source file
mixed_by_source = mixed.groupby("source_file").size().sort_values(ascending=False)
mixed_by_source.head(20)


source_file
/workspaces/protected-bike-lanes-ridership/data/raw/Elliott_Bay_Trail_in_Myrtle_Edwards_Park_Bicycle_and_Pedestrian_Counter_20260112.csv    97347
/workspaces/protected-bike-lanes-ridership/data/raw/MTS_Trail_west_of_I-90_Bridge_Bicycle_and_Pedestrian_Counter_20260112.csv               77963
/workspaces/protected-bike-lanes-ridership/data/raw/Burke_Gilman_Trail_north_of_NE_70th_St_Bicycle_and_Pedestrian_Counter_20260112.csv      57977
/workspaces/protected-bike-lanes-ridership/data/raw/Chief_Sealth_Trail_North_of_Thistle_Bicycle_Counter_(Out_of_Service)_20260112.csv       16054
/workspaces/protected-bike-lanes-ridership/data/raw/2nd_Ave_Cycle_Track_North_of_Marion_St_Bicycle_Counter_20260112.csv                     13391
dtype: int64

In [12]:
# Show a few example mixed rows with the columns that are actually populated
examples = mixed.head(10)
examples

,source_file,date,bike_north,bike_south,bike_east,bike_west,bicyclist_northbound,bicyclist_southbound,bicyclists_northbound,bicyclists_southbound,...,ped_north,ped_south,pedestrian_east,pedestrian_west,pedestrian_northbound,pedestrian_southbound,scooter_east,scooter_west,scooterist_northbound,scooterist_southbound
83040,/workspaces/protected-bike-lanes-ridership/dat...,2024 Jun 22 12:00:00 AM,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
83041,/workspaces/protected-bike-lanes-ridership/dat...,2024 Jun 22 01:00:00 AM,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
83042,/workspaces/protected-bike-lanes-ridership/dat...,2024 Jun 22 02:00:00 AM,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
83043,/workspaces/protected-bike-lanes-ridership/dat...,2024 Jun 22 03:00:00 AM,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
83044,/workspaces/protected-bike-lanes-ridership/dat...,2024 Jun 22 04:00:00 AM,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
83045,/workspaces/protected-bike-lanes-ridership/dat...,2024 Jun 22 05:00:00 AM,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
83046,/workspaces/protected-bike-lanes-ridership/dat...,2024 Jun 22 06:00:00 AM,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
83047,/workspaces/protected-bike-lanes-ridership/dat...,2024 Jun 22 07:00:00 AM,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
83048,/workspaces/protected-bike-lanes-ridership/dat...,2024 Jun 22 08:00:00 AM,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
83049,/workspaces/protected-bike-lanes-ridership/dat...,2024 Jun 22 09:00:00 AM,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0


In [13]:
# Show non-null examples
row = mixed.iloc[0]
row[row.notna()]

source_file              /workspaces/protected-bike-lanes-ridership/dat...
date                                               2024 Jun 22 12:00:00 AM
bicyclists_northbound                                                  0.0
bicyclists_southbound                                                  0.0
scooterist_northbound                                                  0.0
scooterist_southbound                                                  0.0
Name: 83040, dtype: object

In [14]:
# Define a helper function to check if any of the specified columns have a nonzero value, treating NaN as zero. The above example may have floats that are 0.0 but not null, so we want to check for actual nonzero values rather than just non-null values.
def any_nonzero(df, cols):
    if not cols:
        return pd.Series(False, index=df.index)
    return (df[cols].fillna(0) != 0).any(axis=1)

In [15]:
# Repeat the above analysis but checking for actual nonzero counts rather than just non-null values, since some of these may be floats that are 0.0 but not null.
bike_nonzero = any_nonzero(df, bike_cols)
bike_rows = df.loc[bike_nonzero].copy()

ped_nonzero_on_bike = any_nonzero(bike_rows, ped_cols)
scooter_nonzero_on_bike = any_nonzero(bike_rows, scooter_cols)

print("Bike rows (non-zero):", len(bike_rows))
print("Bike rows w/ non-zero ped:", int(ped_nonzero_on_bike.sum()))
print("Bike rows w/ non-zero scooter:", int(scooter_nonzero_on_bike.sum()))
print(
    "Bike rows w/ non-zero ped OR scooter:",
    int((ped_nonzero_on_bike | scooter_nonzero_on_bike).sum())
)

Bike rows (non-zero): 583702
Bike rows w/ non-zero ped: 215733
Bike rows w/ non-zero scooter: 17530
Bike rows w/ non-zero ped OR scooter: 228325


In [16]:
#Extract the mixed rows based on nonzero values rather than just non-null values
mixed_nonzero = bike_rows.loc[
    ped_nonzero_on_bike | scooter_nonzero_on_bike,
    ["source_file", "date"] + bike_cols + ped_cols + scooter_cols
]

In [17]:
# Count mixed rows per source file
mixed_nonzero.groupby("source_file").size().sort_values(ascending=False).head(20)

source_file
/workspaces/protected-bike-lanes-ridership/data/raw/Elliott_Bay_Trail_in_Myrtle_Edwards_Park_Bicycle_and_Pedestrian_Counter_20260112.csv    97278
/workspaces/protected-bike-lanes-ridership/data/raw/Burke_Gilman_Trail_north_of_NE_70th_St_Bicycle_and_Pedestrian_Counter_20260112.csv      57977
/workspaces/protected-bike-lanes-ridership/data/raw/MTS_Trail_west_of_I-90_Bridge_Bicycle_and_Pedestrian_Counter_20260112.csv               55218
/workspaces/protected-bike-lanes-ridership/data/raw/2nd_Ave_Cycle_Track_North_of_Marion_St_Bicycle_Counter_20260112.csv                     12040
/workspaces/protected-bike-lanes-ridership/data/raw/Chief_Sealth_Trail_North_of_Thistle_Bicycle_Counter_(Out_of_Service)_20260112.csv        5812
dtype: int64

In [18]:
# Get a few example mixed rows with populated columns based on nonzero values rather than just non-null values
row = mixed_nonzero.iloc[0]
row[row.notna() & (row != 0)]

source_file              /workspaces/protected-bike-lanes-ridership/dat...
date                                               2024 Jun 23 11:00:00 AM
bicyclists_northbound                                                  4.0
bicyclists_southbound                                                  2.0
scooterist_northbound                                                 13.0
scooterist_southbound                                                 10.0
Name: 83075, dtype: object

In [19]:
# Define generic direction column names that might be totals or transport-agnostic counts (what we're checking for next)
dir_generic_cols = ["north", "south", "east", "west", "nb", "sb"]
dir_generic_cols = [c for c in dir_generic_cols if c in df.columns]
dir_generic_cols

['north', 'south', 'east', 'west', 'nb', 'sb']

In [20]:
# Rebuild the mixed rows logic but now also require at least one of the generic direction columns to be nonzero, since those are more likely to be transport-agnostic counts that could indicate mode mixing rather than just bike-specific counters.
bike_nonzero = any_nonzero(df, bike_cols)
bike_rows = df.loc[bike_nonzero].copy()

ped_nonzero_on_bike = any_nonzero(bike_rows, ped_cols)
scooter_nonzero_on_bike = any_nonzero(bike_rows, scooter_cols)

mixed_mask = ped_nonzero_on_bike | scooter_nonzero_on_bike

# NEW: require at least one of north/south/east/west/nb/sb to be nonzero too
dir_generic_nonzero = any_nonzero(bike_rows, dir_generic_cols)

mixed_with_generic = bike_rows.loc[
    mixed_mask & dir_generic_nonzero,
    ["source_file", "date"] + dir_generic_cols + bike_cols + ped_cols + scooter_cols
]

print("Mixed (nonzero):", int(mixed_mask.sum()))
print("Mixed + generic directional present (nonzero):", len(mixed_with_generic))


Mixed (nonzero): 228325
Mixed + generic directional present (nonzero): 0


In [21]:
# Show a few example mixed rows with populated columns based on nonzero values and generic directional counts, rather than just non-null values. 
# If they're empty, that means there are no rows that have bike + ped/scooter mixing *and* nonzero generic directional counts, which suggests that the mode mixing we're seeing in the earlier analysis may be mostly from files that have bike-specific counters but not more general directional counts.
if mixed_with_generic.empty:
    print("No rows match: mixed-mode + generic directional (nonzero).")
else:
    row = mixed_with_generic.iloc[0]
    display(row[row.notna() & (row != 0)])
    # Show the first 5 examples to see the variety of columns that are populated in these mixed rows with generic directional counts.
    for i in range(5):
        r = mixed_with_generic.iloc[i]
        print("\n--- example", i, "---")
        rint(r[r.notna() & (r != 0)])

No rows match: mixed-mode + generic directional (nonzero).


In [ ]:
# Given the lack of co-occurrence of bike + ped/scooter mixing with nonzero generic directional counts, it's safe to assume the generic counts are from sensors without the ability to differentiate between bikes/peds/scooters and are likely transport-agnostic totals. 
# This means we can still use those generic directional counts for analysis, and we can attribute the bike/ped/scooter-specific counts to their respective modes without worrying too much about mode mixing in those generic fields.